# PRÉDIRE LES ANNULATIONS DE RÉSERVATIONS HÔTELIÈRES

# 3. Machine Learning / Classification

## 3.1 Sélection des variables et prétraitement (Pipeline)

**Moment de la prédiction** : au moment où la réservation est enregistrée. Le modèle n'utilise donc que des informations connues à cet instant.

Variables exclues et pourquoi :
- reservation_status et reservation_status_date : elles contiennent l'issue de la réservation (fuite de données).
- assigned_room_type, booking_changes, days_in_waiting_list, required_car_parking_spaces, total_of_special_requests : elles peuvent changer ou n'être connues qu'après la réservation, voire à l'arrivée.
- country : peut être complété à l'arrivée du client.
- agent et company : ce sont des identifiants, pas des quantités.
- arrival_date_year, week_number et day_of_month : on garde seulement le mois, pour ne pas apprendre des dates précises du passé.

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

# Variables connues au moment de la réservation uniquement
num_cols = [
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "adr",
]
cat_cols = [
    "hotel",
    "arrival_date_month",
    "meal",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "deposit_type",
    "customer_type",
]
features = num_cols + cat_cols
X = df[features]
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)
print(X_train.shape, X_test.shape, round(y_train.mean() * 100, 2))

(69766, 18) (17442, 18) 27.51


- 69 766 réservations pour l'entraînement et 17 442 pour le test, avec 18 variables.
- stratify=y garde la même part d'annulations dans les deux jeux (27,51 % dans le train).
- L'encodage est appris sur le train seulement, puis appliqué au test.

## 3.2 Entraînement et comparaison des modèles

## Modèle 1 : Régression Logistique

In [14]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_prep, y_train)
proba_lr = lr.predict_proba(X_test_prep)[:, 1]
print(f"ROC-AUC Logistic Regression: {roc_auc_score(y_test, proba_lr):.4f}")
print(classification_report(y_test, lr.predict(X_test_prep)))

ROC-AUC Logistic Regression: 0.7483
              precision    recall  f1-score   support

           0       0.76      0.96      0.85     12644
           1       0.68      0.20      0.31      4798

    accuracy                           0.75     17442
   macro avg       0.72      0.58      0.58     17442
weighted avg       0.74      0.75      0.70     17442



## Modèle 2 : Random Forest

In [15]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
rf.fit(X_train_prep, y_train)
proba_rf = rf.predict_proba(X_test_prep)[:, 1]
print(f"ROC-AUC Random Forest: {roc_auc_score(y_test, proba_rf):.4f}")
print(classification_report(y_test, rf.predict(X_test_prep)))

ROC-AUC Random Forest: 0.7761
              precision    recall  f1-score   support

           0       0.75      1.00      0.85     12644
           1       0.90      0.11      0.19      4798

    accuracy                           0.75     17442
   macro avg       0.82      0.55      0.52     17442
weighted avg       0.79      0.75      0.67     17442



## Modèle 3 : Arbre de décision

In [16]:
dt = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
dt.fit(X_train_prep, y_train)
proba_dt = dt.predict_proba(X_test_prep)[:, 1]
print(f"ROC-AUC Decision Tree: {roc_auc_score(y_test, proba_dt):.4f}")
print(classification_report(y_test, dt.predict(X_test_prep)))

ROC-AUC Decision Tree: 0.7487
              precision    recall  f1-score   support

           0       0.74      1.00      0.85     12644
           1       0.98      0.08      0.15      4798

    accuracy                           0.75     17442
   macro avg       0.86      0.54      0.50     17442
weighted avg       0.81      0.75      0.66     17442



## 3.3 Comparaison des modèles en validation croisée

In [17]:
modeles = {
    "Référence (classe majoritaire)": DummyClassifier(),
    "Régression logistique": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Arbre de décision": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE),
}
for nom, modele in modeles.items():
    scores = cross_val_score(make_pipeline(preprocessor, modele), X_train, y_train, cv=5, scoring="roc_auc")
    print(nom, round(scores.mean(), 4), round(scores.std(), 4))

Référence (classe majoritaire) 0.5 0.0


Régression logistique 0.7404 0.0041


Arbre de décision 0.7416 0.0055


Random Forest 0.7691 0.0047


| Modèle | AUC moyenne (validation croisée, 5 plis) | Écart-type |
|---|---|---|
| Référence (classe majoritaire) | 0,5 | 0,0 |
| Régression logistique | 0,7404 | 0,0041 |
| Arbre de décision | 0,7416 | 0,0055 |
| Random Forest | 0,7691 | 0,0047 |

- Les 3 modèles font nettement mieux que la référence (0,5 = hasard).
- Le Random Forest a la meilleure AUC moyenne, et les écarts-types sont faibles : le classement est stable d'un pli à l'autre.
- Sur le test, son AUC est de 0,7761, proche de la validation croisée : pas de surapprentissage marqué.
- **Modèle retenu : Random Forest.**

**Pourquoi l'AUC** : la direction veut classer les réservations de la plus risquée à la moins risquée pour en choisir 100. L'AUC mesure justement la capacité à bien classer. Au seuil par défaut de 0,5, le rappel des annulations est faible (0,11 pour le Random Forest, 0,20 pour la régression logistique, 0,08 pour l'arbre) : on n'utilise donc pas ce seuil, mais un classement par probabilité.

## 3.4 Erreurs du modèle retenu

In [18]:
print(confusion_matrix(y_test, rf.predict(X_test_prep)))

[[12588    56]
 [ 4282   516]]


- 516 annulations bien détectées.
- 4 282 annulations non détectées (faux négatifs) : la chambre risque de rester vide sans que l'hôtel ait été prévenu.
- 56 fausses alertes (faux positifs) : un client qui vient bien est considéré à risque. Si on le contacte trop, il peut être gêné.
- 12 588 réservations honorées bien reconnues.

Au seuil de 0,5, le modèle est très prudent : quand il annonce une annulation, il a souvent raison, mais il en rate beaucoup. C'est pourquoi la règle de priorisation utilise le classement par probabilité plutôt que ce seuil.

# 4. Décision / Recommandations

## 4.1 Règle de priorisation

In [19]:
# Simulation : 1 000 réservations à venir tirées au hasard dans le jeu de test
test = X_test.copy()
test["proba"] = rf.predict_proba(X_test_prep)[:, 1]
test["annule"] = y_test
lot = test.sample(1000, random_state=RANDOM_STATE)

# Priorisation des 100 réservations avec la plus forte probabilité d'annulation
top100 = lot.nlargest(100, "proba")
print(top100["annule"].sum(), round(lot["annule"].mean() * 100, 1))
print(top100["deposit_type"].value_counts())

75 30.3
deposit_type
No Deposit    82
Non Refund    18
Name: count, dtype: int64


- Sur 1 000 réservations tirées au hasard dans le test, le taux d'annulation moyen est de 30,3 %.
- Parmi les 100 réservations les plus risquées selon le modèle, 75 ont réellement été annulées.
- 18 de ces 100 réservations ont un acompte non remboursable (Non Refund). Pour celles-là, l'hôtel garde l'argent en cas d'annulation : les appeler a peu d'intérêt. On teste donc une variante sans elles.

In [20]:
# Variante : on écarte les réservations avec acompte non remboursable (l'hôtel garde déjà l'argent)
top100_sans_acompte = lot[lot["deposit_type"] != "Non Refund"].nlargest(100, "proba")
print(top100_sans_acompte["annule"].sum())

69


## Synthèse et recommandations métier

1. **Règle d'action** : chaque semaine, classer les réservations à venir par probabilité d'annulation (Random Forest), écarter celles avec acompte non remboursable, puis contacter les 100 premières sur 1 000 pour reconfirmer le séjour ou proposer une option de flexibilité.
2. **Justification** : dans la simulation, cette règle cible 69 annulations réelles sur 100 réservations contactées, alors que le taux d'annulation moyen des 1 000 réservations est de 30,3 %. Sans écarter les Non Refund, on cible 75 annulations sur 100, mais 18 de ces réservations ne coûtent rien à l'hôtel si elles sont annulées.
3. **Limites** :
   - données de 2015 à 2017 et deux hôtels seulement : les comportements ont pu changer ;
   - simulation faite sur un seul tirage de 1 000 réservations du jeu de test ;
   - au seuil de 0,5, le modèle rate 4 282 annulations sur le test : il sert à prioriser, pas à prévoir toutes les annulations ;
   - le coût d'un appel et son effet réel ne sont pas connus, et contacter un client peut parfois provoquer l'annulation ;
   - les réglages des modèles (max_depth) n'ont pas été optimisés.